# Stage 6 – Fine-Tuning with New Dataset

In [35]:

import os
import tensorflow as tf

OLD_MODEL_PATH = '/kaggle/working/best_efficientnet_b0_exp50.keras'
NEW_MODEL_PATH = '/kaggle/working/best_efficientnet_b0_finetuned.keras'

model = tf.keras.models.load_model(OLD_MODEL_PATH)

                                
base_model = None

for layer in model.layers:
    if isinstance(layer, tf.keras.Model):
        base_model = layer
        break

if base_model is None:
    raise ValueError("EfficientNet backbone was not found.")

                         
base_model.trainable = False

                                                        
for layer in base_model.layers[-50:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

                                           
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        NEW_MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=16,
    callbacks=callbacks,
    shuffle=True
)


Epoch 1/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.8125 - loss: 0.4561
Epoch 1: val_accuracy improved from None to 0.78581, saving model to /kaggle/working/best_efficientnet_b0_finetuned.keras

Epoch 1: finished saving model to /kaggle/working/best_efficientnet_b0_finetuned.keras
281/281 ━━━━━━━━━━━━━━━━━━━━ 57s 120ms/step - accuracy: 0.8153 - loss: 0.4607 - val_accuracy: 0.7858 - val_loss: 0.6009 - learning_rate: 1.0000e-05
Epoch 2/15
279/281 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8381 - loss: 0.4269
Epoch 2: val_accuracy did not improve from 0.78581
281/281 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.8305 - loss: 0.4365 - val_accuracy: 0.7776 - val_loss: 0.5999 - learning_rate: 1.0000e-05
Epoch 3/15
279/281 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8439 - loss: 0.4070
Epoch 3: val_accuracy did not improve from 0.78581
281/281 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.8409 - loss: 0.4197 - val_accuracy: 0.7831 - val_loss: 0.5866 - learning_rate

In [36]:

import tensorflow as tf

old_model = tf.keras.models.load_model(
    '/kaggle/working/best_efficientnet_b0_exp50.keras'
)

new_model = tf.keras.models.load_model(
    '/kaggle/working/best_efficientnet_b0_finetuned.keras'
)

old_loss, old_acc = old_model.evaluate(
    X_val, y_val, verbose=0
)

new_loss, new_acc = new_model.evaluate(
    X_val, y_val, verbose=0
)

print(f"Old model accuracy: {old_acc * 100:.2f}%")
print(f"New model accuracy: {new_acc * 100:.2f}%")

print(f"Old model loss: {old_loss:.4f}")
print(f"New model loss: {new_loss:.4f}")


Old model accuracy: 77.76%
New model accuracy: 79.81%
Old model loss: 0.6034
New model loss: 0.5804


In [37]:
import os

model_paths = {
    "78_percent": "/kaggle/working/best_efficientnet_b0_exp50.keras",
    "81_percent": "/kaggle/working/best_efficientnet_b0_finetuned.keras"
}

for name, path in model_paths.items():
    if os.path.exists(path):
        print(f"{name}: Found ({os.path.getsize(path) / (1024**2):.2f} MB)")
    else:
        print(f"{name}: NOT FOUND")

78_percent: Found (31.35 MB)
81_percent: Found (39.17 MB)


In [38]:
import os
import shutil

BACKUP_DIR = "/kaggle/working/saved_models"
os.makedirs(BACKUP_DIR, exist_ok=True)

shutil.copy2(
    "/kaggle/working/best_efficientnet_b0_exp50.keras",
    f"{BACKUP_DIR}/DR_EfficientNetB0_78_percent.keras"
)

shutil.copy2(
    "/kaggle/working/best_efficientnet_b0_finetuned.keras",
    f"{BACKUP_DIR}/DR_EfficientNetB0_81_percent.keras"
)

print("Both models backed up successfully!")
print(os.listdir(BACKUP_DIR))

Both models backed up successfully!
['DR_EfficientNetB0_78_percent.keras', 'DR_EfficientNetB0_81_percent.keras']


In [39]:
import os

DATA_PATH = "/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/"

files = [
    "X_train_augmented.npy",
    "y_train_augmented.npy",
    "X_val.npy",
    "y_val.npy"
]

for file in files:
    path = os.path.join(DATA_PATH, file)
    print(f"{file}: {'Found' if os.path.exists(path) else 'NOT FOUND'}")

X_train_augmented.npy: Found
y_train_augmented.npy: Found
X_val.npy: Found
y_val.npy: Found


In [40]:

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

          
DATA_PATH = "/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/"

OLD_MODEL_PATH = "/kaggle/working/best_efficientnet_b0_finetuned.keras"

NEW_MODEL_PATH = "/kaggle/working/DR_EfficientNetB0_new_data.keras"

                         
X_train = np.load(DATA_PATH + "X_train_augmented.npy")
y_train = np.load(DATA_PATH + "y_train_augmented.npy")

X_val = np.load(DATA_PATH + "X_val.npy")
y_val = np.load(DATA_PATH + "y_val.npy")

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Train pixel range:", X_train.min(), X_train.max())

                                        
model = tf.keras.models.load_model(OLD_MODEL_PATH)

                                     
base_model = None

for layer in model.layers:
    if "efficientnetb0" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise ValueError(
        "EfficientNetB0 backbone not found. "
        "Print model.summary() to inspect the architecture."
    )

                                 
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

for layer in base_model.layers[-50:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

                                          
                           
                                                                 

BATCH_SIZE = 16

def prepare_image(image, label):
    image = tf.cast(image, tf.float32) * 255.0
    label = tf.cast(label, tf.int32)
    return image, label

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train), reshuffle_each_iteration=True)
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

            
model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

                                                               
callbacks = [
    ModelCheckpoint(
        NEW_MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

          
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)

print("Training finished.")
print("New model path:", NEW_MODEL_PATH)


Train: (4489, 224, 224, 3) (4489,)
Validation: (733, 224, 224, 3) (733,)
Train pixel range: 0.0 1.0
Epoch 1/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.8596 - loss: 0.3720
Epoch 1: val_accuracy improved from None to 0.78718, saving model to /kaggle/working/DR_EfficientNetB0_new_data.keras

Epoch 1: finished saving model to /kaggle/working/DR_EfficientNetB0_new_data.keras
281/281 ━━━━━━━━━━━━━━━━━━━━ 58s 116ms/step - accuracy: 0.8568 - loss: 0.3806 - val_accuracy: 0.7872 - val_loss: 0.5886 - learning_rate: 5.0000e-06
Epoch 2/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8626 - loss: 0.3723
Epoch 2: val_accuracy did not improve from 0.78718
281/281 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.8605 - loss: 0.3771 - val_accuracy: 0.7872 - val_loss: 0.5976 - learning_rate: 5.0000e-06
Epoch 3/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8636 - loss: 0.3644
Epoch 3: val_accuracy did not improve from 0.78718

Epoch 3: ReduceLROnPlateau reducing lear